In [ ]:
# ============ 공통 기준점 재계산: L5B·L6B·L7 모든 모델을 같은 코드 loss 지점에서 비교 ============
import glob, json, numpy as np, pandas as pd

def fi(folder, name):
    h = glob.glob(f'/kaggle/input/**/{folder}/{name}', recursive=True)
    assert h, f'{folder}/{name} 없음 — Add Input 확인'
    return h[0]

C5 = json.load(open(fi('exp_l5_finetune', 'ft_curves.json')))
C6 = json.load(open(fi('exp_l6_finetune', 'ft_curves.json')))
C7 = json.load(open(fi('exp_l7', 'ft_curves.json')))

# (이름, 곡선 묶음, 모델 키, 기준 off 이름, 비용)
SRC = [('off (L6A)',        C7, 'pre_off_repro',          None,        1.00),
       ('decay ρ0.02',      C6, 'pre_decay_spec_0.02',    'off (L6A)', 1.26),
       ('decay ρ0.05',      C6, 'pre_decay_spec_0.05',    'off (L6A)', 1.26),
       ('decay ρ0.1',       C7, 'pre_decay_spec_0.1',     'off (L6A)', 1.24),
       ('full ρ0.05',       C7, 'pre_full_spec_0.05',     'off (L6A)', 2.27),
       ('off (L5A)',        C5, 'pre_off',                None,        1.00),
       ('decay ρ0.008',     C5, 'pre_decay_spec_0.008',   'off (L5A)', 1.26),
       ('full ρ0.008',      C5, 'pre_full_spec',          'off (L5A)', 2.29)]
ADAM = {'weak': 1e-4, 'strong': 1e-3 / 3}

def get(C, pre, lr, s):
    for k, v in C.items():
        a, ss, p = k.split('|')
        if p == pre and int(ss) == s and abs(float(a) - lr) < 1e-9: return v
    raise KeyError((pre, lr, s))

def baby_at(curve, target):
    for a, b in zip(curve, curve[1:]):
        if b['code'] <= target:
            t = 1.0 if a['code'] == b['code'] else min(max((a['code'] - target) / (a['code'] - b['code']), 0), 1)
            return a['baby'] + t * (b['baby'] - a['baby'])
    return float('nan')

rows = []
for st, lr in ADAM.items():
    cv = {name: [get(C, pre, lr, s) for s in (0, 1, 2)] for name, C, pre, _, _ in SRC}
    target = max(min(p['code'] for p in c) for cs in cv.values() for c in cs)   # 8개 모델 × 3 seed 모두 도달한 지점
    mf = {name: np.array([baby_at(c, target) - c[0]['baby'] for c in cs]) for name, cs in cv.items()}
    print(f'\n===== [{st}] 공통 기준 코드 loss = {target:.3f} =====')
    for name, C, pre, base, cost in SRC:
        code_after = np.mean([c[-1]['code'] for c in cv[name]])
        if base is None:
            print(f'  {name:14s} 기준 | 같은 지점 망각 {mf[name].mean():.4f} | 코드 loss 후 {code_after:.3f}')
            continue
        d = mf[base] - mf[name]
        red = 1 - mf[name].mean() / mf[base].mean()
        rows.append(dict(strength=st, model=name, cost=cost, reduction=red, pos=int((d > 0).sum()), code_after=code_after))
        print(f'  {name:14s} 비용 {cost:.2f}배 | 망각 감소 {red*100:+5.1f}% ({int((d>0).sum())}/3) | 코드 loss 후 {code_after:.3f}')

df = pd.DataFrame(rows)
print('\n===== 헤드라인: 같은 기준점에서 decay ρ0.05 vs full =====')
for st in ADAM:
    r = df[df.strength == st].set_index('model').reduction
    print(f'  [{st}] decay ρ0.05 {r["decay ρ0.05"]*100:+.1f}% | full ρ0.05 {r["full ρ0.05"]*100:+.1f}% '
          f'→ decay가 full의 {r["decay ρ0.05"]/r["full ρ0.05"]*100:.0f}% | 참고 full ρ0.008 {r["full ρ0.008"]*100:+.1f}%')
df.to_csv('/kaggle/working/common_target.csv', index=False)